In [0]:
from pyspark.sql import functions as F

TABELA_ORIGEM = "voebem.bronze.vra"
TABELA_DESTINO = "voebem.silver.vra"

bronze = spark.table(TABELA_ORIGEM)

In [0]:
COLUNAS_DATA = ["partida_prevista", "partida_real", "chegada_prevista", "chegada_real"]
FORMATO = "yyyy-MM-dd HH:mm:ss"

com_datas = bronze
for col in COLUNAS_DATA:
    com_datas = com_datas.withColumn(
        col,
        F.expr(f"try_to_timestamp({col}, '{FORMATO}')")
    )

In [0]:
com_datas.printSchema()

In [0]:
com_limpo = com_datas.withColumn(
    "codigo_justificativa",
    F.when(F.col("codigo_justificativa") == "N/A", None)
    .otherwise(F.col("codigo_justificativa"))
)

In [0]:
com_limpo = com_limpo.withColumn(
    "situacao_voo",
    F.upper(F.trim(F.col("situacao_voo")))
)

In [0]:
com_limpo.filter(F.col("codigo_justificativa") == "N/A").count()

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS voebem.silver")

In [0]:
(
    com_limpo.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"{TABELA_DESTINO}: {spark.table(TABELA_DESTINO).count():,} linhas")

In [0]:
spark.table("voebem.bronze.vra").count()